In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


##### lag features

In [0]:
agg_df = spark.table("fmcg.gold.fact_sales_monthly_agg")

# Window: partition by the series identity, order by time to put the data in chronological order.
w = Window.partitionBy("product_code", "channel").orderBy("date")

feat_df = (
    agg_df
    .withColumn("lag_1", F.lag("sold_quantity", 1).over(w)) #the sales quantity from the previous row(previous month)
    .withColumn("lag_2", F.lag("sold_quantity", 2).over(w))# Sales from 2 months ago.
    .withColumn("lag_3", F.lag("sold_quantity", 3).over(w)) #Sales from 3 months ago.
    .withColumn("lag_12", F.lag("sold_quantity", 12).over(w))  # same month, last year bc we expect yearly seasonality
    .withColumn("rolling_mean_3", F.avg("sold_quantity").over(w.rowsBetween(-3, -1))) # the average sales of the previous 3 rows/months.
    .withColumn("month", F.month("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("year", F.year("date"))
)

display(feat_df.orderBy("product_code", "channel", "date").limit(30))

product_code,channel,date,sold_quantity,division,category,product,price_inr,market,platform,lag_1,lag_2,lag_3,lag_12,rolling_mean_3,month,quarter,year
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-07-01,76219,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar,null,null,null,null,null,7,3,2025
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-08-01,75681,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar,76219,null,null,null,76219.0,8,3,2025
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-09-01,80900,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar,75681,76219,null,null,75950.0,9,3,2025
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-10-01,75013,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar,80900,75681,76219,null,77600.0,10,4,2025
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-11-01,71409,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar,75013,80900,75681,null,77198.0,11,4,2025
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-12-01,9061,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar,71409,75013,80900,null,75774.0,12,4,2025
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Acquisition,2025-07-01,124782,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),100.0,India,Sports Bar,null,null,null,null,null,7,3,2025
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Acquisition,2025-08-01,120725,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),100.0,India,Sports Bar,124782,null,null,null,124782.0,8,3,2025
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Acquisition,2025-09-01,121121,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),100.0,India,Sports Bar,120725,124782,null,null,122753.5,9,3,2025
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Acquisition,2025-10-01,114977,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),100.0,India,Sports Bar,121121,120725,124782,null,122209.33333333333,10,4,2025


##### Drop rows where lag features can't be computed (no history yet)

In [0]:
feat_df_clean = feat_df.filter(F.col("lag_3").isNotNull())
feat_df_clean.count(), feat_df.count()

(12315, 14700)

In [0]:
(feat_df_clean.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fmcg.gold.fact_sales_features")
)